In [ ]:
"""
exp13 Step 01: Stress Test Of The Frozen E1M1 On 2005-2014 (PROTOCOL.md §4, step 01)

STRESS = the first session where both the 200-session average and vix_level_prev exist (on or after 2005-10-04) ->
2014-12-31, the only span where E1M1 has never run. SPY bars and every VIX load are cut at
so.config.EXPLORATION_DATA_CUTOFF_DATE_STR (2015-03-18): nothing of 2015-2026 is read.

The frozen E1M1 (exp12's code), the unmodified E1 and E2M1 (reported alongside) and buy-and-hold, each as one continuous
path (start invested, no forced buy-back) over three windows: STRESS as a whole; 2007-10-01 -> 2009-08-12 (the 2008 bear
market); 2009-08-13 -> 2014-12-31 (exp12's exploration span: E1M1 +104.51% and E1 +49.00% are asserted). Then the full
episode scorecard of E1M1 over STRESS (VIX at the exit, maximum while out, at the buy-back; for every M1 re-entry the
lowest close until the next exit and the drawdown from the buy-back price to it).

E1M1 was chosen after the fact on 2015-2026; this step cannot make that out-of-sample evidence. Caveat: the 2008 bear
market and its VIX peak are known to everyone. Outputs: store04_experiments/exp13_e1m1_robustness/step01_stress_data/.
Trials: 2 (stage "exploration": E1M1, E2M1).
"""
None

In [ ]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp13_e1m1_robustness import config as exp_config
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT EXIT AND RE-ENTRY SIMULATION FUNCTIONS
from so.core.reentry_simulation import get_buy_and_hold_result_dict
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp13_e1m1_robustness.robustness import get_span_data_dict, check_last_date_dict, get_stress_start_date, get_year_period_pdf, get_frozen_rule_tuple, \
    replay_rule_dict, get_path_metric_dict, get_stress_scorecard_pdf, check_step_trial_dict
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [ ]:
# CHECK THE REAL DATA ONLY OUTSIDE THE SYNTHETIC SMOKE RUN (scripts/smoke_notebook.py)
CHECK_REAL_DATA_BOOL = __name__ != "__smoke__"
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step01_stress_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "exploration")

In [ ]:
# CALL FUNCTION TO READ SPY AND THE VIX UP TO THE STRESS CUTOFF (NOTHING OF 2015-2026 IS READ) AND BUILD THE FEATURES
cutoff_date_str = exp_config.STRESS_DATA_CUTOFF_DATE_STR
span_data_dict = get_span_data_dict(cutoff_date_str)
ohlcv_array_dict, daily_pdf, vix_feature_pdf = span_data_dict["ohlcv_array_dict"], span_data_dict["daily_pdf"], span_data_dict["vix_feature_pdf"]
# DISPLAY AND ASSERT THE LAST DATES LOADED
print(f"SPY sessions loaded:\t{len(daily_pdf):,}\tfrom {span_data_dict['first_spy_date']}")
last_date_str_dict = check_last_date_dict(span_data_dict["last_date_dict"], cutoff_date_str, CHECK_REAL_DATA_BOOL)
# DEFINE THE STRESS START AND THE THREE WINDOWS
stress_start_date = get_stress_start_date(daily_pdf, vix_feature_pdf)
window_dict = {"stress": (str(stress_start_date), exp_config.STRESS_END_DATE_STR), "bear_2007_2009": exp_config.BEAR_WINDOW_TUPLE,
               "exp12_exploration": exp_config.EXP12_EXPLORATION_WINDOW_TUPLE}
for window_str, (date1_str, date2_str) in window_dict.items():
    window_date_list = [d for d in ohlcv_array_dict["session_date_list"] if pd.Timestamp(date1_str).date() <= d <= pd.Timestamp(date2_str).date()]
    print(f"Window {window_str}:\t{window_date_list[0]} -> {window_date_list[-1]}\t({len(window_date_list):,} sessions)")
    if CHECK_REAL_DATA_BOOL and window_str != "stress":
        assert str(window_date_list[0]) == date1_str and str(window_date_list[-1]) == date2_str, f"❌ Window {window_str} differs"

In [ ]:
"""
E1M1, the unmodified E1 and E2M1 (exp12's code), each one continuous path per window, against buy-and-hold over the same
sessions. The exp12 exploration window reproduces exp12 step 01 (asserted).
"""
# DEFINE THE PATHS
candidate_dict_list = [exp_config.FROZEN_CANDIDATE_DICT, exp_config.UNMODIFIED_E1_CANDIDATE_DICT, exp_config.E2M1_CANDIDATE_DICT]
# LISTS TO HOLD THE ROWS AND THE SIMULATIONS OF THE STRESS WINDOW
row_dict_list, stress_simulation_dict = [], {}
# ITERATE OVER THE WINDOWS
for window_str, (date1_str, date2_str) in window_dict.items():
    # SIMULATE BUY AND HOLD AND DEFINE THE YEARS AS PERIODS
    buy_hold_dict = get_buy_and_hold_result_dict(ohlcv_array_dict, date1_str, date2_str)
    year_period_pdf = get_year_period_pdf(ohlcv_array_dict["session_date_list"], date1_str, date2_str)
    row_dict_list.append({"window": window_str, "candidate_name": "buy_hold", "first_date": date1_str, "last_date": date2_str,
                          "total_return": buy_hold_dict["metric_dict"]["total_return"], "buy_hold_total_return": buy_hold_dict["metric_dict"]["total_return"],
                          "excess_total_return": 0.0, "sharpe_ratio": buy_hold_dict["metric_dict"]["sharpe_ratio"], "max_drawdown": buy_hold_dict["metric_dict"]["max_drawdown"],
                          "in_market_pct": 1.0, "exit_count": 0})
    # ITERATE OVER THE PATHS
    for candidate_dict in candidate_dict_list:
        simulation_dict = replay_rule_dict(ohlcv_array_dict, get_frozen_rule_tuple(daily_pdf, vix_feature_pdf, candidate_dict), date1_str, date2_str)
        row_dict_list.append({"window": window_str, "candidate_name": candidate_dict["candidate_name"], **get_path_metric_dict(simulation_dict, buy_hold_dict, year_period_pdf, ohlcv_array_dict)})
        if window_str == "stress":
            stress_simulation_dict[candidate_dict["candidate_name"]] = simulation_dict
stress_pdf = pd.DataFrame(row_dict_list)
# ASSERT THE REPRODUCTION OF exp12 STEP 01 (2009-08-13 -> 2014-12-31)
reproduction_pdf = stress_pdf[stress_pdf["window"] == "exp12_exploration"].set_index("candidate_name")
reproduction_dict = {name: round(100 * reproduction_pdf.loc[name, "total_return"], 2) for name in ["E1M1", "E1none", "buy_hold"]}
print(f"Reproduction of exp12 step 01:\t{reproduction_dict}")
if CHECK_REAL_DATA_BOOL:
    assert reproduction_dict["E1M1"] == exp_config.REPRODUCTION_EXPLORATION_E1M1_TOTAL_PCT, "❌ E1M1 differs from exp12 step 01"
    assert reproduction_dict["E1none"] == exp_config.REPRODUCTION_EXPLORATION_E1_TOTAL_PCT, "❌ E1 differs from exp12 step 01"
    assert reproduction_dict["buy_hold"] == exp_config.REPRODUCTION_EXPLORATION_BUY_HOLD_PCT, "❌ Buy-and-hold differs from exp12 step 01"
    print("Reproduction asserts:\tPASSED")
# PREVIEW DATAFRAME
stress_pdf[["window", "candidate_name", "first_date", "last_date", "total_return", "buy_hold_total_return", "excess_total_return", "annualized_log_excess",
            "log_excess_ci_low_6m", "log_excess_ci_high_6m", "log_excess_ci_low_1m", "log_excess_ci_high_1m", "in_market_pct", "exit_count",
            "sharpe_ratio", "max_drawdown", "product_s_over_r"]].round(4)

In [ ]:
"""
Episode scorecard of E1M1 over STRESS. S / R > 1 = bought back lower than sold; vix_at_exit / vix_max_out /
vix_at_reentry = vix_level_prev at the exit decision, its maximum while out, at the buy-back decision; for an M1
re-entry, lowest_close_to_next_exit and drawdown_to_low_pct = SPY's lowest close until the next exit and the fall from
the buy-back price to it.
"""
# BUILD THE SCORECARDS
scorecard_pdf_dict = {name: get_stress_scorecard_pdf(simulation_dict, ohlcv_array_dict, daily_pdf, vix_feature_pdf) for name, simulation_dict in stress_simulation_dict.items()}
e1m1_scorecard_pdf = scorecard_pdf_dict["E1M1"]
scorecard_col_list = ["exit_date", "exit_fill_price", "reentry_date", "reentry_fill_price", "s_over_r", "sessions_out", "reentry_trigger", "vix_at_exit", "vix_max_out",
                      "vix_at_reentry", "lowest_close_to_next_exit", "drawdown_to_low_pct", "buy_hold_return_out", "max_decline_out_pct", "open_at_end"]
# DISPLAY THE SUMMARY OF EVERY PATH
for name, scorecard_pdf in scorecard_pdf_dict.items():
    reason_dict = scorecard_pdf["reentry_trigger"].value_counts().to_dict() if len(scorecard_pdf) else {}
    print(f"{name}:\t{len(scorecard_pdf)} episodes, {int((scorecard_pdf['s_over_r'] > 1).sum()) if len(scorecard_pdf) else 0} bought back lower\t"
          f"product of S/R {float(np.prod(scorecard_pdf['s_over_r'])) if len(scorecard_pdf) else 1.0:.4f}\treasons {reason_dict}")
# DISPLAY THE FULL SCORECARD OF E1M1
e1m1_scorecard_pdf[scorecard_col_list].round(4)

In [ ]:
# DISPLAY THE E1M1 EPISODES OF 2007-2009
print("E1M1 episodes with an exit in 2007-2009:")
e1m1_scorecard_pdf[[2007 <= d.year <= 2009 for d in e1m1_scorecard_pdf["exit_date"]]][scorecard_col_list].round(4) if len(e1m1_scorecard_pdf) else e1m1_scorecard_pdf

In [ ]:
# SAVE THE RESULTS
write_csv_file_to_path(stress_pdf, f"{output_path_str}stress_path_data.csv", "W")
write_csv_file_to_path(pd.concat([scorecard_pdf.assign(candidate_name=name) for name, scorecard_pdf in scorecard_pdf_dict.items()], ignore_index=True),
                       f"{output_path_str}stress_scorecard_data.csv", "W")

In [ ]:
# CHECK THE TRIAL BUDGET AND THE EXPECTED COUNT BEFORE LOGGING
trial_log_pdf = read_trial_log_pdf()
check_step_trial_dict(trial_log_pdf, "step01", 2)
# LOG E1M1 AND E2M1 AS EXPLORATION TRIALS (THEIR METRICS OVER THE THREE WINDOWS; WINDOW COLUMNS = STRESS)
logged_dict_list = []
for candidate_dict in [exp_config.FROZEN_CANDIDATE_DICT, exp_config.E2M1_CANDIDATE_DICT]:
    metric_dict = {f"{row['window']}_{key}": value for _, row in stress_pdf[stress_pdf["candidate_name"] == candidate_dict["candidate_name"]].iterrows()
                   for key, value in row.items() if key not in ["window", "candidate_name"]}
    metric_dict.update({f"{window_str}_buy_hold_total_return": float(stress_pdf[(stress_pdf["window"] == window_str) & (stress_pdf["candidate_name"] == "buy_hold")]["total_return"].iloc[0])
                        for window_str in window_dict})
    logged_dict_list.append(log_trial_dict(exp_config, "exploration", {"step": "step01", "candidate_name": candidate_dict["candidate_name"], "exit_rule": candidate_dict["exit_rule"],
                                                                       "modification": candidate_dict["modification"], "span": "stress"}, metric_dict, False,
                                           fold_row_in={"valid_start": window_dict["stress"][0], "valid_end": window_dict["stress"][1]},
                                           note_str_in="exp13 step01 stress test 2005-2014 (frozen exp12 rule, continuous replay)"))
print(f"Trials logged:\t{len(logged_dict_list)}\tfirst logged_ts {logged_dict_list[0]['logged_ts']}\ttest_window_touched: {sorted({row['test_window_touched'] for row in logged_dict_list})}")

In [ ]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)